# Phase 11/12: I-JEPA Continuation Scaling Locally

Run this notebook on a local CUDA GPU. It keeps Phase 11 reproducible and adds Phase 12 as the lean kill test for tuned I-JEPA continuation.

Phase 11 tested:

- One-epoch weight sweep: `jepa_weight` 0.05, 0.1, 0.2 at `lr=5e-5`
- Longer continuation sweep: `jepa_weight=0.05`, epochs 2 and 3 at `lr=5e-5`

Phase 12 tests:

- Tuned longer continuation: `jepa_weight` 0.1 and 0.2, epochs 2 and 3 at `lr=5e-5`
- Matched DINO+MAE controls for each LR/epoch setting
- Winner-only broader victim panel: current victims plus `efficientnet_b0` and `swin_t`
- Seeds: 0, 1, 2
- Train limit: 1000 Imagenette train images
- Eval limit: 5000, covering full Imagenette validation
- Loss weights normalized with `--normalize-loss-weights`

The notebook uses `--skip-existing`, training AMP with GradScaler, eval AMP, and generator `torch.compile(..., mode="reduce-overhead")`. Evaluation victim compilation is left off by default on local 8 GB GPUs because it can spike memory.


## Setup

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), 'A local CUDA GPU is required for this experiment.'

In [ ]:
from pathlib import Path

cwd = Path.cwd()
candidates = [cwd, cwd.parent]
REPO_ROOT = next(
    (path for path in candidates if (path / 'scripts' / 'sweep_dsva_jepa_finetune.py').exists()),
    None,
)
assert REPO_ROOT is not None, f'Could not find repo root from {cwd}'
print('Repo root:', REPO_ROOT)

In [ ]:
TRAIN_ROOT = REPO_ROOT / 'imagenette2-320' / 'train'
VAL_ROOT = REPO_ROOT / 'imagenette2-320' / 'val'
DSVA_CHECKPOINT = REPO_ROOT / 'external' / 'models' / 'dSVA' / 'model.pth'

assert TRAIN_ROOT.exists(), f'Missing training data: {TRAIN_ROOT}'
assert VAL_ROOT.exists(), f'Missing validation data: {VAL_ROOT}'
assert DSVA_CHECKPOINT.exists(), f'Missing dSVA checkpoint: {DSVA_CHECKPOINT}'

print('Train root:', TRAIN_ROOT)
print('Val root:', VAL_ROOT)
print('dSVA checkpoint:', DSVA_CHECKPOINT)

## Configuration

In [ ]:
SEEDS = [0, 1, 2]
TRAIN_LIMIT = 1000
EVAL_LIMIT = 5000
LR = '0.00005'
EPSILON = '0.06274509803921569'
OUTPUT_MODE = 'scaled-delta'
CORE_VICTIMS = ['resnet50', 'convnext_tiny', 'vit_b_16']
BROADER_VICTIMS = [*CORE_VICTIMS, 'efficientnet_b0', 'swin_t']

PHASE11_OUTPUT_DIR = REPO_ROOT / 'results' / 'phase11_jepa_scale'
PHASE11_ANALYSIS_DIR = REPO_ROOT / 'results' / 'phase11_analysis'
PHASE12_OUTPUT_DIR = REPO_ROOT / 'results' / 'phase12_jepa_scale'
PHASE12_ANALYSIS_DIR = REPO_ROOT / 'results' / 'phase12_analysis'
PHASE12_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE12_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PERF_FLAGS = ['--compile', '--compile-mode', 'reduce-overhead']
EVAL_PERF_FLAGS = ['--eval-amp', *TRAIN_PERF_FLAGS]
# Keep eval victim compilation disabled by default on local 8 GB GPUs.
# Add '--eval-compile' to EVAL_PERF_FLAGS only if you have enough VRAM.

PHASE11_WEIGHT_CONFIGS = ['0.05:0.00005', '0.1:0.00005', '0.2:0.00005']
PHASE11_LONGER_RUNS = [
    {'label': 'jw0p05_ep2', 'configs': ['0.05:0.00005'], 'epochs': 2},
    {'label': 'jw0p05_ep3', 'configs': ['0.05:0.00005'], 'epochs': 3},
]
PHASE12_LONGER_RUNS = [
    {'label': 'jw0p1_0p2_ep2', 'configs': ['0.1:0.00005', '0.2:0.00005'], 'epochs': 2},
    {'label': 'jw0p1_0p2_ep3', 'configs': ['0.1:0.00005', '0.2:0.00005'], 'epochs': 3},
]

print('Phase 11 output dir:', PHASE11_OUTPUT_DIR)
print('Phase 12 output dir:', PHASE12_OUTPUT_DIR)
print('Performance flags:', EVAL_PERF_FLAGS)
print('Core victims:', CORE_VICTIMS)
print('Broader victims:', BROADER_VICTIMS)


## Optional Smoke Test

In [ ]:
import subprocess, sys

smoke_cmd = [
    sys.executable,
    'scripts/run_dsva_checkpoint_attack.py',
    '--data-root', str(VAL_ROOT),
    '--checkpoint', str(DSVA_CHECKPOINT),
    '--output-mode', 'adv',
    '--limit', '16',
    '--batch-size', '8',
    '--epsilon', EPSILON,
    '--victims', *CORE_VICTIMS,
    '--device', 'cuda',
    '--output-csv', 'results/phase12_smoke_released_dsva.csv',
    '--amp',
]
print(' '.join(smoke_cmd))
subprocess.run(smoke_cmd, cwd=REPO_ROOT, check=True)


## Optional Phase 11 Reproduction: One-Epoch JEPA Weight Sweep


In [ ]:
import subprocess, sys
from pathlib import Path

def run_checked(cmd):
    print('\n' + ' '.join(map(str, cmd)), flush=True)
    log_dir = REPO_ROOT / 'results' / 'phase12_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (Path(str(cmd[1])).stem + '_' + str(len(list(log_dir.glob('*.log')))).replace(' ', '_') + '.log')
    with log_path.open('w', encoding='utf-8', errors='replace') as handle:
        result = subprocess.run(
            cmd,
            cwd=REPO_ROOT,
            text=True,
            stdout=handle,
            stderr=subprocess.STDOUT,
        )
    text = log_path.read_text(encoding='utf-8', errors='replace')
    print(text[-12000:])
    print('Log:', log_path)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


for seed in SEEDS:
    seed_dir = PHASE11_OUTPUT_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'scripts/sweep_dsva_jepa_finetune.py',
        '--train-root', str(TRAIN_ROOT),
        '--val-root', str(VAL_ROOT),
        '--init-checkpoint', str(DSVA_CHECKPOINT),
        '--output-dir', str(seed_dir),
        '--run-prefix', f'phase11_seed{seed}_weight_sweep',
        '--configs', *PHASE11_WEIGHT_CONFIGS,
        '--limit', str(TRAIN_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--epochs', '1',
        '--batch-size', '1',
        '--eval-batch-size', '8',
        '--grad-accum-steps', '8',
        '--epsilon', EPSILON,
        '--output-mode', OUTPUT_MODE,
        '--victims', *CORE_VICTIMS,
        '--device', 'cuda',
        '--seed', str(seed),
        '--normalize-loss-weights',
        '--skip-existing',
        *EVAL_PERF_FLAGS,
    ]
    run_checked(cmd)


## Optional Phase 11 Reproduction: Longer Continuation Sweep


In [ ]:
for run in PHASE11_LONGER_RUNS:
    for seed in SEEDS:
        seed_dir = PHASE11_OUTPUT_DIR / f'seed_{seed}'
        seed_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            sys.executable,
            'scripts/sweep_dsva_jepa_finetune.py',
            '--train-root', str(TRAIN_ROOT),
            '--val-root', str(VAL_ROOT),
            '--init-checkpoint', str(DSVA_CHECKPOINT),
            '--output-dir', str(seed_dir),
            '--run-prefix', f"phase11_seed{seed}_{run['label']}",
            '--configs', *run['configs'],
            '--limit', str(TRAIN_LIMIT),
            '--eval-limit', str(EVAL_LIMIT),
            '--epochs', str(run['epochs']),
            '--batch-size', '1',
            '--eval-batch-size', '8',
            '--grad-accum-steps', '8',
            '--epsilon', EPSILON,
            '--output-mode', OUTPUT_MODE,
            '--victims', *CORE_VICTIMS,
            '--device', 'cuda',
            '--seed', str(seed),
            '--normalize-loss-weights',
            '--skip-existing',
            *EVAL_PERF_FLAGS,
        ]
        run_checked(cmd)


## Aggregate Phase 11 Results


In [ ]:
import pandas as pd

run_checked([
    sys.executable,
    'scripts/analyze_phase11_scale_results.py',
    '--scale-root', 'results/phase11_jepa_scale',
    '--output-detail-csv', 'results/phase11_analysis/jepa_scale_detail.csv',
    '--output-aggregate-csv', 'results/phase11_analysis/jepa_scale_aggregate.csv',
])

pd.read_csv(PHASE11_ANALYSIS_DIR / 'jepa_scale_aggregate.csv')


## Phase 12: Tuned Longer Continuation


In [ ]:
for run in PHASE12_LONGER_RUNS:
    for seed in SEEDS:
        seed_dir = PHASE12_OUTPUT_DIR / f'seed_{seed}'
        seed_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            sys.executable,
            'scripts/sweep_dsva_jepa_finetune.py',
            '--train-root', str(TRAIN_ROOT),
            '--val-root', str(VAL_ROOT),
            '--init-checkpoint', str(DSVA_CHECKPOINT),
            '--output-dir', str(seed_dir),
            '--run-prefix', f"phase12_seed{seed}_{run['label']}",
            '--configs', *run['configs'],
            '--limit', str(TRAIN_LIMIT),
            '--eval-limit', str(EVAL_LIMIT),
            '--epochs', str(run['epochs']),
            '--batch-size', '1',
            '--eval-batch-size', '8',
            '--grad-accum-steps', '8',
            '--epsilon', EPSILON,
            '--output-mode', OUTPUT_MODE,
            '--victims', *CORE_VICTIMS,
            '--device', 'cuda',
            '--seed', str(seed),
            '--normalize-loss-weights',
            '--skip-existing',
            *EVAL_PERF_FLAGS,
        ]
        run_checked(cmd)


## Aggregate Phase 12 Tuned Results


In [ ]:
run_checked([
    sys.executable,
    'scripts/analyze_phase11_scale_results.py',
    '--scale-root', 'results/phase12_jepa_scale',
    '--output-detail-csv', 'results/phase12_analysis/jepa_scale_detail.csv',
    '--output-aggregate-csv', 'results/phase12_analysis/jepa_scale_aggregate.csv',
])

phase12_aggregate = pd.read_csv(PHASE12_ANALYSIS_DIR / 'jepa_scale_aggregate.csv')
phase12_aggregate


## Pick Phase 12 Winner


In [ ]:
def pick_best_jepa_config(aggregate):
    rows = aggregate.copy()
    rows['mean_transfer_success'] = rows['mean_transfer_success'].astype(float)
    controls = rows[rows['run_type'] == 'control'].set_index(['lr', 'epochs', 'train_limit'])['mean_transfer_success']
    candidates = rows[rows['run_type'] == 'jepa'].copy()
    candidates['control_mean'] = candidates.apply(
        lambda row: controls.loc[(row['lr'], row['epochs'], row['train_limit'])],
        axis=1,
    )
    candidates['matched_gain'] = candidates['mean_transfer_success'] - candidates['control_mean']
    return candidates.sort_values(['matched_gain', 'mean_transfer_success'], ascending=False).iloc[0]

best = pick_best_jepa_config(phase12_aggregate)
BEST_JEPA_WEIGHT = str(best['jepa_weight'])
BEST_LR = str(best['lr'])
BEST_EPOCHS = int(best['epochs'])
BEST_CONFIG = f'{BEST_JEPA_WEIGHT}:{BEST_LR}'
print('Best tuned config:', BEST_CONFIG, 'epochs:', BEST_EPOCHS, 'matched gain:', f"{best['matched_gain']:.4f}")
best.to_frame().T


## Phase 12: Winner-Only Broader Victim Panel


In [ ]:
for seed in SEEDS:
    seed_dir = PHASE12_OUTPUT_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'scripts/sweep_dsva_jepa_finetune.py',
        '--train-root', str(TRAIN_ROOT),
        '--val-root', str(VAL_ROOT),
        '--init-checkpoint', str(DSVA_CHECKPOINT),
        '--output-dir', str(seed_dir),
        '--run-prefix', f'phase12_seed{seed}_broader_victims_ep{BEST_EPOCHS}_jw{BEST_JEPA_WEIGHT.replace(".", "p")}',
        '--configs', BEST_CONFIG,
        '--limit', str(TRAIN_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--epochs', str(BEST_EPOCHS),
        '--batch-size', '1',
        '--eval-batch-size', '8',
        '--grad-accum-steps', '8',
        '--epsilon', EPSILON,
        '--output-mode', OUTPUT_MODE,
        '--victims', *BROADER_VICTIMS,
        '--device', 'cuda',
        '--seed', str(seed),
        '--normalize-loss-weights',
        '--skip-existing',
        *EVAL_PERF_FLAGS,
    ]
    run_checked(cmd)


## Aggregate Phase 12 Final Results


In [ ]:
run_checked([
    sys.executable,
    'scripts/analyze_phase11_scale_results.py',
    '--scale-root', 'results/phase12_jepa_scale',
    '--output-detail-csv', 'results/phase12_analysis/jepa_scale_detail.csv',
    '--output-aggregate-csv', 'results/phase12_analysis/jepa_scale_aggregate.csv',
])

pd.read_csv(PHASE12_ANALYSIS_DIR / 'jepa_scale_aggregate.csv')


## Package CSV Outputs

In [ ]:
import tarfile

archive_path = REPO_ROOT / 'results' / 'phase12_csv_artifacts.tar.gz'
paths = list(PHASE11_OUTPUT_DIR.rglob('*.csv'))
paths += list(PHASE11_OUTPUT_DIR.rglob('*.json'))
paths += list(PHASE11_ANALYSIS_DIR.rglob('*.csv'))
paths += list(PHASE12_OUTPUT_DIR.rglob('*.csv'))
paths += list(PHASE12_OUTPUT_DIR.rglob('*.json'))
paths += list(PHASE12_ANALYSIS_DIR.rglob('*.csv'))
for smoke_name in ['phase11_smoke_released_dsva.csv', 'phase12_smoke_released_dsva.csv']:
    smoke_csv = REPO_ROOT / 'results' / smoke_name
    if smoke_csv.exists():
        paths.append(smoke_csv)

with tarfile.open(archive_path, 'w:gz') as tar:
    for path in paths:
        tar.add(path, arcname=path.relative_to(REPO_ROOT))
print('Archive:', archive_path)
